In [ ]:
# Written by tools/build_notebooks.py -- do not edit. The bootstrap cell below
# compares this against the repo it clones and tells you if these cells are old.
CELLS_SRC = "kaggle_05_demo.py"
CELLS_SHA = "e726ef431e023c3d"

# 05 — The demo: audio in, one track per person out

**Accelerator: None (CPU) is enough.** **Runtime: ~2 minutes.** Costs **no GPU quota**.

Every other notebook reports an average over 1,500 mixtures. This one runs a single file and
lets you *listen* to it, because an average cannot be played out loud and a viva can.

It is also the only notebook that uses the system as a system. Notebook 02 trains the counter,
notebook 03 trains the separator, notebook 04 scores them — this is the one that joins them:

```
  3 s of audio ──┬──►  counter    ──►  N̂ = "3 people"
                 └──►  separator  ──►  5 slots + noise
                                             │
                           keep the N̂ loudest┘ ──►  speaker_01.wav, speaker_02.wav, ...
```

## Why the predicted count is printed here, and was not in the old version

The old project's demo **deliberately deleted** its "N speakers detected" line. Its counting
head answered **1** for 1445 of 1500 test mixtures, so that line printed the same number
whoever was talking — a headline that looks like a result and carries no information.

In this version the counter is a separate model with its own gradient, so the number means
something and is printed. It comes with the full probability distribution, so a clip the model
is unsure about *looks* unsure instead of confidently wrong.

## Before you press Run

1. **+ Add Input → Notebook Output →** notebook 00 (the store and the recipes)
2. **+ Add Input → Notebook Output →** notebook 02 (the counter checkpoint)
3. **+ Add Input → Notebook Output →** notebook 03 (the separator checkpoint)
4. **Settings → Accelerator → None** — this is forward passes on one file; a GPU here is
   30 hours a week spent on nothing.

## Bootstrap (this cell is identical in every notebook)

Three ways to get the code onto the Kaggle machine, tried in order:

1. **GitHub clone** — set `REPO_URL` below and turn *Internet* ON in the notebook
   settings panel (Settings → Internet → On). This is the recommended route.
2. **Repo-as-dataset** — upload this folder as a Kaggle Dataset called
   `speaker-count-separate-v1` and attach it. No internet needed. Use this if your
   account cannot enable internet (phone-verification is required for that).
3. **Already there** — an existing clone is **fast-forwarded to the newest commit**,
   not reused as-is. A Kaggle session outlives many pushes, and silently running code
   from an hour ago is the most expensive kind of confusion: the log looks fine and the
   fix you are testing is not in it. Any local edits inside the clone are discarded.

Whichever route runs, the commit is printed. Every log can then be traced to the exact
code that produced it.

In [ ]:
REPO_URL = "https://github.com/AlAminAshraf01/speaker-count-separate-v1.git"
REPO_DIR = "/kaggle/working/speaker-count-separate-v1"
REPO_AS_DATASET = "/kaggle/input/speaker-count-separate-v1"

import hashlib
import os
import shutil
import subprocess
import sys


def cells_fingerprint(src_dir: str, name: str) -> str:
    """Short hash of one notebook's percent source plus this shared bootstrap.

    ``tools/build_notebooks.py`` stamps this into every generated ``.ipynb``. The copy
    running on Kaggle recomputes it from the freshly-cloned repo, so a notebook whose
    cells were imported before the last push says so in the first ten seconds instead of
    eleven hours later.

    Line endings are normalised first. The same file is CRLF in a Windows working tree
    and LF in a Linux clone, and a fingerprint that disagrees with itself across
    platforms is worse than no fingerprint at all.
    """
    digest = hashlib.sha256()
    for part in (name, "_bootstrap.py"):
        with open(os.path.join(src_dir, part), "rb") as fh:
            digest.update(fh.read().replace(b"\r\n", b"\n"))
        digest.update(b"\0")
    return digest.hexdigest()[:16]


def cells_status(repo_dir: str, src_name: str | None, stamp: str | None) -> str:
    """Compare the stamp baked into these cells with the repo they are about to run.

    Never raises. A check that can take down every notebook is a worse bug than the one
    it detects, so anything unreadable degrades to "cannot verify".
    """
    if not src_name or not stamp:
        return "unstamped -- re-import this notebook to enable the staleness check"
    try:
        current = cells_fingerprint(os.path.join(repo_dir, "notebooks", "src"), src_name)
    except Exception as exc:
        return f"cannot verify ({exc})"
    if current == stamp:
        return f"current ({stamp})"
    return "\n".join([
        f"STALE  cells {stamp} but repo has {current}",
        "",
        "  These notebook cells were imported before the newest push, so the fix you",
        "  are about to test is not in them. scripts/ and src/ just updated themselves;",
        "  notebook cells cannot, because Kaggle owns them.",
        "",
        "  Fix: File -> Import Notebook -> upload notebooks/" + src_name[:-3] + ".ipynb",
        "       again, re-attach the inputs, and re-run.",
    ])


def _git(repo_dir: str, *argv: str) -> subprocess.CompletedProcess:
    return subprocess.run(["git", "-C", repo_dir, *argv],
                          capture_output=True, text=True)


def update_clone(repo_dir: str) -> str:
    """Fast-forward an existing clone to the remote's newest commit.

    Returns a short status for printing; never raises. Losing internet is a reason to
    carry on with the code that is already there, but it is not a reason to be quiet
    about it -- running stale code unknowingly is how a fix gets tested without being
    present.
    """
    if not os.path.isdir(os.path.join(repo_dir, ".git")):
        return "not a git clone, left as it is"
    branch = _git(repo_dir, "rev-parse", "--abbrev-ref", "HEAD").stdout.strip() or "main"
    before = _git(repo_dir, "rev-parse", "--short", "HEAD").stdout.strip()
    fetched = _git(repo_dir, "fetch", "--depth", "1", "origin", branch)
    if fetched.returncode != 0:
        tail = (fetched.stderr or "").strip().splitlines()
        return f"COULD NOT FETCH ({tail[-1] if tail else 'unknown'}) -- code may be stale"
    reset = _git(repo_dir, "reset", "--hard", f"origin/{branch}")
    if reset.returncode != 0:
        tail = (reset.stderr or "").strip().splitlines()
        return f"COULD NOT UPDATE ({tail[-1] if tail else 'unknown'}) -- code may be stale"
    after = _git(repo_dir, "rev-parse", "--short", "HEAD").stdout.strip()
    return "already newest" if before == after else f"updated {before} -> {after}"


def describe_commit(repo_dir: str) -> str:
    """``<short sha> <date> <subject>`` for the checked-out commit, or a plain note."""
    out = _git(repo_dir, "log", "-1", "--format=%h %cs %s").stdout.strip()
    return out or "no git metadata"


def bootstrap(repo_url: str = REPO_URL, repo_dir: str = REPO_DIR) -> str:
    """Put the repo at `repo_dir`, put its `src/` on sys.path, and chdir into it."""
    if not os.path.isdir(os.path.join(repo_dir, "src")):
        if os.path.isdir(os.path.join(REPO_AS_DATASET, "src")):
            shutil.copytree(REPO_AS_DATASET, repo_dir, dirs_exist_ok=True)
            print(f"copied repo from the attached dataset {REPO_AS_DATASET}")
        else:
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_dir], check=True)
            print(f"cloned {repo_url}")
    else:
        print(f"existing clone: {update_clone(repo_dir)}")
    src = os.path.join(repo_dir, "src")
    if src not in sys.path:
        sys.path.insert(0, src)
    os.chdir(repo_dir)
    return repo_dir


REPO = bootstrap()

import countsep  # noqa: E402

print("countsep", countsep.__version__, "at", REPO)
print("code ", describe_commit(REPO))
# CELLS_SRC / CELLS_SHA are set by the stamp cell that tools/build_notebooks.py puts at
# the top of every generated notebook. globals().get keeps this working in a notebook
# assembled by hand, where that cell may not exist.
CELLS = cells_status(REPO, globals().get("CELLS_SRC"), globals().get("CELLS_SHA"))
print("cells", CELLS if "\n" not in CELLS else "")
if "\n" in CELLS:
    print(CELLS)
print("python", sys.version.split()[0])

import torch  # noqa: E402

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| devices", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  [{i}] {p.name}  {p.total_memory / 1e9:.1f} GB")

In [ ]:
import shlex
import time


def run(cmd: str, check: bool = True) -> int:
    """Run a shell command, streaming its output into the notebook."""
    print("$", cmd, flush=True)
    t0 = time.time()
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    print(f"\n[exit {code} in {time.time() - t0:.1f}s]", flush=True)
    if check and code != 0:
        raise SystemExit(f"command failed with exit code {code}")
    return code

In [ ]:
import glob
import sys
import time
sys.path.insert(0, os.path.join(REPO, "scripts"))
from _common import autodetect_store, find_recipes

STORE = autodetect_store()
RECIPES_TEST = find_recipes("recipes_test.csv", STORE)

# Pin either of these to a path to force a particular run; otherwise the newest
# attached checkpoint wins and every candidate is printed.
COUNTER_OVERRIDE = None
SEPARATOR_OVERRIDE = None


def _find(pattern, override=None):
    """The NEWEST matching checkpoint, with every candidate shown.

    Alphabetical order has nothing to do with which model you meant, and once you have
    re-run a trainer there are two of everything attached. Notebook 04 had the same bug.
    """
    if override:
        print(f"    using the pinned path: {override}")
        return override
    hits = glob.glob(pattern, recursive=True)
    if not hits:
        return None
    hits.sort(key=os.path.getmtime, reverse=True)
    if len(hits) > 1:
        print(f"    {len(hits)} candidates matched {pattern} -- taking the newest:")
        for i, h in enumerate(hits):
            when = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(h)))
            print(f"      {'->' if i == 0 else '  '} {when}  {h}")
        print("      (set COUNTER_OVERRIDE / SEPARATOR_OVERRIDE above to pin one instead)")
    return hits[0]

COUNTER   = (_find("/kaggle/input/**/counter/ckpt/best.pt", COUNTER_OVERRIDE)
             or _find("/kaggle/input/**/ckpt/best.pt"))
SEPARATOR = _find("/kaggle/input/**/sep/ckpt/best.pt", SEPARATOR_OVERRIDE)

print("store     :", STORE)
print("recipes   :", RECIPES_TEST)
print("counter   :", COUNTER or "MISSING")
print("separator :", SEPARATOR or "MISSING")
if COUNTER is None or SEPARATOR is None:
    raise SystemExit(
        "This notebook needs BOTH models -- it is the one that joins them.\n"
        "'+ Add Input' -> 'Notebook Output' -> notebook 02, then again for notebook 03.")
if STORE is None or RECIPES_TEST is None:
    raise SystemExit("Attach notebook 00's output: '+ Add Input' -> 'Notebook Output'.")

run(f"python scripts/preflight.py --for eval --store {STORE}"
    f" --recipes_test {RECIPES_TEST} --ckpt {COUNTER}"
    f" --cells_src {CELLS_SRC} --cells_sha {CELLS_SHA}")

## Run it on mixtures whose true count you already know

These are drawn from the **frozen test set**, so the notebook can print *predicted vs true*
rather than asking you to take the number on faith. Two clips: an easy one and a hard one.

**What to look at in the output, in order:**

1. **The probability bars.** A model that is right for the right reason puts most of its mass
   on one class. Mass spread over 4 and 5 on a 5-speaker clip is a near-miss; mass parked on
   1 regardless of the input is the old project's failure, and you would see it immediately.

2. **The slot power table.** The separator always emits 5 slots. Training pushes the unused
   ones toward −30 dB, so on a 3-speaker clip you want a clear cliff after slot 3. If the
   powers slope gently instead, the two models disagree about the count — and you can hear
   which one is right.

In [ ]:
DEMOS = [(2, "two talkers -- the easy case"), (4, "four talkers -- the hard case")]

for n, label in DEMOS:
    print("\n" + "=" * 74 + f"\n{label}\n" + "=" * 74)
    run(f"python scripts/08_infer.py"
        f" --counter {COUNTER} --separator {SEPARATOR}"
        f" --store {STORE} --from_recipes {RECIPES_TEST}"
        f" --n {n} --index 0 --all_slots"
        f" --out /kaggle/working/demo_n{n}")

## Listen

The mixture first, then one track per person the system decided was there. The surplus slots
are included so you can hear what "rejected" sounds like — they should be near-silence or a
faint smear, not a fourth voice.

All files share **one** gain, so the tracks keep their relative loudness. Normalising each
track separately would make a whisper and a shout come out the same, which would hide exactly
the thing the slot-power table is showing you.

In [ ]:
import json
from IPython.display import Audio, display

for n, label in DEMOS:
    out_dir = f"/kaggle/working/demo_n{n}"
    with open(os.path.join(out_dir, "infer_report.json")) as fh:
        rep = json.load(fh)
    print("\n" + "=" * 74)
    print(f"{label}   true {rep['true_n']}  ->  predicted {rep['n_hat']}"
          f"   ({'correct' if rep['true_n'] == rep['n_hat'] else 'WRONG'})")
    print("=" * 74)
    for name in rep["files"]:
        path = os.path.join(out_dir, name)
        if not os.path.exists(path):
            continue
        print(f"\n{name}")
        display(Audio(filename=path))

## Your own recording

**+ Add Input → Upload → New Dataset**, upload any audio file, then put its path below and
run the cell. Any format and any sample rate — it is converted to 8 kHz mono internally.

Two honest warnings before you read too much into the result:

**The models were trained on fully-overlapped speech.** Every training clip has all N people
talking at once for the full three seconds. A normal conversation is mostly people *taking
turns*, and during a single-speaker stretch the honest answer is "1" even though three people
are in the room. That is not a bug, it is a different problem — diarisation — and it is the
first thing an examiner will poke at.

**Long files are processed in 3-second windows**, because both models were trained at that
length; feeding one long block instead took the old counter from a working score to **0
correct out of 300**. Windows are permutation-aligned to their neighbour before being
cross-faded together — without that step a speaker changes track partway through the file,
which was measured here at 0.908 correlation with speaker A over the first half of a clip and
1.000 with speaker B over the second.

In [ ]:
MY_FILE = None      # e.g. "/kaggle/input/my-audio/meeting.m4a"

if MY_FILE:
    run(f"python scripts/08_infer.py --counter {COUNTER} --separator {SEPARATOR}"
        f" --input {MY_FILE} --all_slots --max_seconds 60"
        f" --out /kaggle/working/demo_mine")
    for path in sorted(glob.glob("/kaggle/working/demo_mine/*.wav")):
        print("\n" + os.path.basename(path))
        display(Audio(filename=path))
else:
    print("set MY_FILE above to run this on your own recording")

## What this notebook is worth in the report

One figure and one sentence, not a page.

**The figure:** the slot-power table for a clip the system got right. It shows the cliff after
slot N̂ — the separator's own output agreeing with the counter's answer, which is two
independently trained models arriving at the same number. That is a stronger statement than
either accuracy figure alone, and the old version could not make it at all: its count head
read the separator's own features, so agreement between them was guaranteed by construction
rather than earned.

**The sentence:** quote a clip the system got *wrong*, and say what the probability bars
looked like. A project that shows a failure case and explains it reads as one where somebody
checked; a project with only successes reads as one where somebody stopped looking.

**Save Version → Save & Run All (Commit)** to keep the audio in the output.